# RQ2 — fresh seed-6 policy decomposition (CPU only)

Read-only diagnostic on F10/F50/F100. It compares Uniform, raw log-FLOPs Resource, Pure-SW, frozen Resource, frozen Resource+SW, and the gradient oracle. This notebook never loads a checkpoint, runs a model, refits a predictor, or authorizes Gate C.

In [ ]:
import os, subprocess, sys, json, shutil, zipfile, importlib
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)
print('CPU-only: no CUDA/GPU check and no checkpoint/model import.')

## Resolve immutable inputs

The resolver scans `/kaggle/input` for one unique seed-6 pair/Gram artifact and one unique frozen predictor. Identical duplicate copies are deduplicated by hash. If they are inside a ZIP, only the small diagnostic files are extracted—never `.pt` checkpoints.

In [ ]:
import rq2_fresh_seed6_policy_decomposition as decomposition
decomposition = importlib.reload(decomposition)
FRESH_ROOT, FROZEN_PREDICTOR = decomposition.find_fresh_seed6_inputs(
    Path('/kaggle/input'), '/kaggle/working/materialized-fresh-seed6-policy-decomposition'
)
print('Fresh diagnostic root:', FRESH_ROOT)
print('Frozen predictor:', FROZEN_PREDICTOR)
assert all((decomposition._gram_path(FRESH_ROOT, epoch) is not None) for epoch in (10,50,100))
print('Input audit PASS: pair table + F10/F50/F100 Grams + frozen predictor.')

## Run the six-policy decomposition

`raw_resource` is parameter-free squared log-FLOPs distance. `frozen_resource` and `frozen_resource_plus_sw` use the untouched development coefficients. All policies use the same fixed uniform marginals.

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/fresh-seed6-policy-decomposition')
summary = decomposition.run_fresh_seed6_policy_decomposition(
    FRESH_ROOT, FROZEN_PREDICTOR, OUTPUT_DIR
)
print(json.dumps(summary, indent=2))
assert summary['checkpoint_loaded'] is False
assert summary['predictors_refit'] is False
assert summary['gpu_required'] is False
assert summary['gate_c_authorized'] is False

In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown
wide = pd.read_csv(OUTPUT_DIR/'fresh_seed6_policy_decomposition.csv')
display(wide)
display(pd.read_csv(OUTPUT_DIR/'fresh_seed6_sw_gradient_correlations.csv'))
display(pd.read_csv(OUTPUT_DIR/'fresh_seed6_policy_l1_to_oracle.csv').query("policy in ['raw_resource','pure_sw','frozen_resource_plus_sw']"))
display(Image(filename=str(OUTPUT_DIR/'fresh_seed6_policy_variance.png')))
display(Image(filename=str(OUTPUT_DIR/'fresh_seed6_policy_l1_to_oracle.png')))
display(Markdown((OUTPUT_DIR/'fresh_seed6_policy_decomposition_summary.md').read_text()))

## Validate and export the small CPU result bundle

In [ ]:
required = [
 'fresh_seed6_policy_decomposition.csv',
 'fresh_seed6_policy_variance_long.csv',
 'fresh_seed6_pair_policy_assignments.csv',
 'fresh_seed6_policy_l1_to_oracle.csv',
 'fresh_seed6_sw_gradient_correlations.csv',
 'fresh_seed6_policy_decomposition_summary.json',
 'fresh_seed6_policy_decomposition_summary.md',
 'fresh_seed6_policy_variance.png',
 'fresh_seed6_policy_l1_to_oracle.png',
]
missing = [name for name in required if not (OUTPUT_DIR/name).is_file() or (OUTPUT_DIR/name).stat().st_size == 0]
assert not missing, f'Missing outputs: {missing}'
bundle = Path('/kaggle/working/fresh-seed6-policy-decomposition.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file(): archive.write(path, Path(OUTPUT_DIR.name)/path.relative_to(OUTPUT_DIR))
print('Download/persist:', bundle, f'{bundle.stat().st_size/2**20:.2f} MiB')
bundle